# Non-separable data: affine score or MLP

This notebook accompanies Exercise 5.3. A small dataset of six points reveals two distinct limitations: a loss alone does not determine the boundary, and an affine score cannot represent every separation.

We compare an affine SVM, an affine logistic classifier and an MLP. The final comparison uses the same logistic loss to isolate the effect of the function class.

## Outline

1. [A dataset not separable by an affine score](#jeu-nonseparable)
2. [Two affine criteria](#criteres-affines)
3. [An exact non-affine score](#score-non-affine)
4. [An MLP with Flax NNX](#mlp-nonseparable)
5. [Comparing loss and function class separately](#comparaison-nonseparable)

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax
import flax
from flax import nnx

print(f"JAX {jax.__version__}, Flax {flax.__version__}, Optax {optax.__version__}")

In [ ]:
x_ns = jnp.array([
    [-2.0, -1.0],
    [-2.0,  1.0],
    [ 2.0,  0.0],
    [ 2.0, -1.0],
    [ 2.0,  1.0],
    [-2.0,  0.0],
])
z_ns = jnp.array([-1.0, -1.0, -1.0, 1.0, 1.0, 1.0])

def score_affine(parametres, x):
    return x @ parametres["w"] + parametres["b"]

def marges_affines(parametres, x, z):
    return z * score_affine(parametres, x)

<a id="jeu-nonseparable"></a>
## 1. A dataset not separable by an affine score

1. Plot the data.
2. Examine the three points with first coordinate $2$. Why does an affine score make their exact classification impossible?
3. Repeat the argument for the points with first coordinate $-2$.

In [ ]:
# Complete this section.

<a id="criteres-affines"></a>
## 2. Two affine criteria

Fit the same affine score using:

- regularised hinge loss;
- regularised logistic loss.

Compare error frequency, each loss value and signed margins. Do these three quantities necessarily rank the solutions in the same order?

The training function is provided to keep the experiment focused on comparing criteria.

In [ ]:
def critere_affine(parametres, x, z, lamb, nature):
    marges = marges_affines(parametres, x, z)
    if nature == 0:
        perte = jnp.mean(jax.nn.relu(1.0 - marges))
    else:
        perte = jnp.mean(jax.nn.softplus(-marges))
    return perte + 0.5 * lamb * jnp.sum(parametres["w"]**2)

def entrainer_affine(x, z, lamb, nature, *, iterations=3000, alpha=2.0e-2):
    parametres = {"w": jnp.zeros(x.shape[1]), "b": jnp.array(0.0)}
    transformation = optax.adam(alpha)
    etat = transformation.init(parametres)

    @jax.jit
    def pas(parametres, etat):
        fonction = lambda p: critere_affine(p, x, z, lamb, nature)
        valeur, gradient = jax.value_and_grad(fonction)(parametres)
        mises_a_jour, etat = transformation.update(gradient, etat, parametres)
        return optax.apply_updates(parametres, mises_a_jour), etat, valeur

    for _ in range(iterations):
        parametres, etat, valeur = pas(parametres, etat)
    return parametres, float(valeur)

def statistiques(scores, z):
    marges = z * scores
    return {
        "error frequency": float(jnp.mean(marges <= 0.0)),
        "logistic loss": float(jnp.mean(jax.nn.softplus(-marges))),
        "hinge loss": float(jnp.mean(jax.nn.relu(1.0 - marges))),
        "minimum margin": float(jnp.min(marges)),
    }

In [ ]:
# Complete this section.

<a id="score-non-affine"></a>
## 3. An exact non-affine score

Verify that
$$
F(x_1,x_2)=x_1(2x_2^2-1)
$$
classifies all six points correctly. Plot its decision boundary and explain why it cannot be that of an affine classifier.

In [ ]:
# Complete this section.

In [ ]:
# Complete this section.

<a id="mlp-nonseparable"></a>
## 4. An MLP with Flax NNX

Construct a scalar-valued MLP with two hidden layers and fit it using logistic loss. Penalise the Euclidean norm of all its parameters:
$$
\mathcal L(p)=\frac1{n_{\mathcal D}}\sum_i
\log(1+e^{-z_i\Phi(x_i,p)})+\frac\lambda2\lVert p\rVert^2.
$$

Compare several initialisations if the solution found does not immediately classify all six points.

In [ ]:
# Complete this section.

<a id="comparaison-nonseparable"></a>
## 5. Comparing loss and function class separately

Plot on a single figure:

- the affine SVM;
- the affine score fitted with logistic loss;
- the MLP fitted with the same logistic loss.

Comparing the last two models isolates the effect of the function class. Comparing the first two isolates the effect of the loss.

In [ ]:
# Complete this section.

In [ ]:
# Complete this section.

## Summary

Analysing a classification experiment requires separating at least two choices: the objective function and the machine class. Using the same loss for the affine score and MLP is the most direct way to reveal the second choice.